In [18]:
from pathlib import Path
from PIL import Image
import pandas as pd
import torchvision
from torchvision import transforms
import numpy as np
import torch
from torchvision.transforms import v2
from torchvision.io import decode_image
batch_size = 32

In [19]:
img_path = Path(r"train\1.jpg")
img = Image.open(img_path)
print(img.size)

(256, 256)


In [20]:
train = pd.read_csv(r"train.csv")
validation = pd.read_csv(r"val.csv")
test = pd.read_csv(r"test.csv")
train.head()
X_train = train.drop('label',axis=1)
Y_train = train['label']
X_val = validation.drop('label',axis = 1)
Y_val = validation['label']
X_test = test['id']

In [22]:
transform = v2.Compose([
    v2.ToImage(),                      
    v2.ToDtype(torch.float32, scale=True),  
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

img_tensor = transform(img)
print(img_tensor.shape)
print(img_tensor.dtype)
print(img_tensor.min())
print(img_tensor.max())

torch.Size([3, 256, 256])
torch.float32
tensor(-2.1179)
tensor(2.6400)


In [23]:
from torch.utils.data import Dataset

class SateliteDataset(Dataset):
    def __init__(self,dataframe, image_dir, transform):
        self.dataframe = dataframe
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe['id'])
    
    def __getitem__(self, idx):
        name = self.dataframe.loc[idx,'id']
        label = self.dataframe.loc[idx,'label']
        img = Image.open(self.image_dir / name)
        if self.transform is not None:
            return self.transform(img), label
        else:
            return img, label

In [31]:
transform = v2.Compose([
    v2.ToImage(),                      
    v2.ToDtype(torch.float32, scale=True),  
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
Train_dataset = SateliteDataset(train, r"train",transform=transform)
sample = Train_dataset[0]
print(type(sample))
print(type(sample[0]))
print(sample[0].shape)
print(sample[0].dtype)
print(sample[1])

<class 'tuple'>
<class 'torchvision.tv_tensors._image.Image'>
torch.Size([3, 256, 256])
torch.float32
4


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset=Train_dataset,
    batch_size=32,
    shuffle=True,
)
val_loader = DataLoader(
    dataset = SateliteDataset(validation, r"val", transform=transform),
    batch_size = 32,
    shuffle = False,
)
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 256, 256])
torch.Size([32])


In [49]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, num_classes=20):
        super().__init__()

        self.flatten = nn.Flatten()

        self.fc = nn.Sequential(
            nn.Linear(3 * 64 * 64, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.flatten(x)
        x = self.fc(x)
        return x


In [66]:
MLP_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((64,64)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

Val_dataset = SateliteDataset(validation, r"val", transform=MLP_transform)

Train_dataset = SateliteDataset(train, r"train",transform=MLP_transform)

train_loader = DataLoader(
    dataset=Train_dataset,
    batch_size=32,
    shuffle=True,
)

val_loader = DataLoader(
    dataset = Val_dataset,
    batch_size = 32,
    shuffle= False,
)

In [69]:
model = MLP()
criterion  = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
running_loss = 0.0

model.train()
num_epochs = 10

for epoch in range(num_epochs):
    running_loss = 0.0
    model.train()
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        output = model(images)

        loss_train = criterion(output, labels)

        loss_train.backward()
        optimizer.step()

        running_loss += loss_train.item()
    average_loss = running_loss / len(train_loader)

    model.eval()

    correct = 0

    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            output = model(images)

            predictions = output.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    accuracy = correct / total
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            output = model(images)
            predictions = output.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    train_accuracy = correct / total

    print("Train Accuracy:", train_accuracy)
    print("Validation Accuracy:", accuracy)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {average_loss:.4f}")

Train Accuracy: 0.21084216843368675
Validation Accuracy: 0.13733905579399142
Epoch 1/10, Loss: 3.6105
Train Accuracy: 0.3044608921784357
Validation Accuracy: 0.19027181688125894
Epoch 2/10, Loss: 2.7494
Train Accuracy: 0.3694738947789558
Validation Accuracy: 0.19456366237482117
Epoch 3/10, Loss: 2.3476
Train Accuracy: 0.4344868973794759
Validation Accuracy: 0.16165951359084407
Epoch 4/10, Loss: 2.1061
Train Accuracy: 0.5307061412282457
Validation Accuracy: 0.2088698140200286
Epoch 5/10, Loss: 2.0119
Train Accuracy: 0.5737147429485897
Validation Accuracy: 0.2145922746781116
Epoch 6/10, Loss: 1.7675
Train Accuracy: 0.5215043008601721
Validation Accuracy: 0.1759656652360515
Epoch 7/10, Loss: 1.5974
Train Accuracy: 0.5767153430686137
Validation Accuracy: 0.21602288984263232
Epoch 8/10, Loss: 1.5727
Train Accuracy: 0.6081216243248649
Validation Accuracy: 0.2117310443490701
Epoch 9/10, Loss: 1.4226
Train Accuracy: 0.6795359071814363
Validation Accuracy: 0.22317596566523606
Epoch 10/10, Loss:

Как видим MLP очень плохо справляется с задачами классификации картинок, поэтому вместо подбор гиперпараметров перейдем сразу к CNN

In [83]:
class CNN(nn.Module):
    def __init__(self, num_classes=20):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.flt = nn.Flatten()

        self.fc = nn.Sequential(
            nn.Linear(64 * 32 * 32, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flt(x)
        x = self.fc(x)
        return x

In [84]:
model = CNN()


CNN_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.RandomHorizontalFlip(),
    v2.RandomRotation(10),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

Val_dataset = SateliteDataset(validation, r"val", transform=CNN_transform)

Train_dataset = SateliteDataset(train, r"train",transform=CNN_transform)

train_loader = DataLoader(
    dataset=Train_dataset,
    batch_size=32,
    shuffle=True,
)
val_loader = DataLoader(
    dataset=Val_dataset,
    batch_size=32,
    shuffle=False,
)

images, labels = next(iter(train_loader))
output = model(images)
print(images.shape)
print(output.shape)

torch.Size([32, 3, 256, 256])
torch.Size([32, 20])


In [85]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001,weight_decay=1e-4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

num_epochs = 10

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        output = model(images)

        loss_train = criterion(output, labels)

        loss_train.backward()
        optimizer.step()

        running_loss += loss_train.item()

    average_loss = running_loss / len(train_loader)

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            output = model(images)

            predictions = output.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    accuracy = correct / total
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            output = model(images)
            predictions = output.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    train_accuracy = correct / total

    print("Train Accuracy:", train_accuracy)
    print("Validation Accuracy:", accuracy)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {average_loss:.4f}")

Train Accuracy: 0.2666533306661332
Validation Accuracy: 0.24892703862660945
Epoch 1/10, Loss: 2.7547
Train Accuracy: 0.3530706141228246
Validation Accuracy: 0.30472103004291845
Epoch 2/10, Loss: 2.3771
Train Accuracy: 0.4406881376275255
Validation Accuracy: 0.3748211731044349
Epoch 3/10, Loss: 2.1598
Train Accuracy: 0.4216843368673735
Validation Accuracy: 0.36909871244635195
Epoch 4/10, Loss: 2.0410
Train Accuracy: 0.5057011402280456
Validation Accuracy: 0.44206008583690987
Epoch 5/10, Loss: 1.8973
Train Accuracy: 0.5183036607321464
Validation Accuracy: 0.4663805436337625
Epoch 6/10, Loss: 1.8472
Train Accuracy: 0.5547109421884376
Validation Accuracy: 0.48497854077253216
Epoch 7/10, Loss: 1.7244
Train Accuracy: 0.5615123024604921
Validation Accuracy: 0.4792560801144492
Epoch 8/10, Loss: 1.6471
Train Accuracy: 0.6085217043408682
Validation Accuracy: 0.5078683834048641
Epoch 9/10, Loss: 1.5937
Train Accuracy: 0.632126425285057
Validation Accuracy: 0.5021459227467812
Epoch 10/10, Loss: 1.

In [90]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 20)
ResNet_transform = weights.transforms()
Val_dataset = SateliteDataset(validation, r"val", transform=ResNet_transform)

Train_dataset = SateliteDataset(train, r"train",transform=ResNet_transform)

train_loader = DataLoader(
    Train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    Val_dataset,
    batch_size=32,
    shuffle=False
)

images, labels = next(iter(train_loader))
print(images.shape)

images, labels = next(iter(val_loader))
print(images.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])


In [91]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        output = model(images)

        loss = criterion(output, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(train_loader)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            output = model(images)

            predictions = output.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    accuracy = correct / total

    print(
        f"Epoch {epoch + 1}/{num_epochs}, "
        f"Loss: {average_loss:.4f}, "
        f"Validation Accuracy: {accuracy:.4f}"
    )

Epoch 1/10, Loss: 1.2046, Validation Accuracy: 0.6223
Epoch 2/10, Loss: 0.6846, Validation Accuracy: 0.6009
Epoch 3/10, Loss: 0.4978, Validation Accuracy: 0.7339
Epoch 4/10, Loss: 0.3389, Validation Accuracy: 0.7754
Epoch 5/10, Loss: 0.3176, Validation Accuracy: 0.7454
Epoch 6/10, Loss: 0.2680, Validation Accuracy: 0.7725
Epoch 7/10, Loss: 0.2119, Validation Accuracy: 0.7940
Epoch 8/10, Loss: 0.1879, Validation Accuracy: 0.7725
Epoch 9/10, Loss: 0.1639, Validation Accuracy: 0.8083
Epoch 10/10, Loss: 0.0891, Validation Accuracy: 0.7883
